## GPT-2 step by step

> **Origin and license.** This notebook is derived from [`ch04/01_main-chapter-code/ch04.ipynb`](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/01_main-chapter-code/ch04.ipynb) in the repository [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Copyright 2023-2026 Sebastian Raschka, licensed under the [Apache License 2.0](../LICENSES/Apache-2.0-LLMs-from-scratch.txt).
>
> **Modified by L. Saetta (2026):** explanations added, comments corrected, a parameter count added, and code adapted to this repository. See [NOTICE](../NOTICE).

In [`dummy_gpt.ipynb`](dummy_gpt.ipynb) the model had the right data flow,
but its transformer blocks and layer normalization were placeholders. This
notebook builds the real components, one at a time, and then assembles them
into a complete GPT-2-like model:

1. the configuration;
2. multi-head causal attention, from Chapter 2;
3. layer normalization;
4. the GELU activation;
5. the feed-forward network;
6. the transformer block, which combines them with residual connections;
7. the GPT model, a stack of transformer blocks between the embeddings and the
   output head.

The [architecture diagram](../images/gpt-architecture.png) shows the same
classes, with the tensor shapes at each step.

Every component maps a tensor of shape `(batch_size, num_tokens, emb_dim)` to a
tensor of the **same shape**. This is what makes it possible to stack the
transformer blocks: the output of one block is a valid input for the next.

In [1]:
import torch
from torch import nn

### The configuration

The dictionary collects the hyperparameters of the model. These values are the
dimensions of GPT-2 Small:

* `vocab_size`: the number of tokens of the GPT-2 BPE tokenizer; it sets the
  size of the token embedding table and of the output head.
* `context_length`: the maximum number of tokens that the model reads at once;
  it sets the size of the position embedding table and of the causal mask.
* `emb_dim`: the size of the vector that represents each token, everywhere in
  the model.
* `n_heads`: the number of attention heads; `emb_dim` must be divisible by it,
  so each head works on `768 / 12 = 64` dimensions.
* `n_layers`: the number of transformer blocks in the stack.
* `drop_rate`: the dropout probability, used in three places: after the
  embeddings, on the attention weights, and on the residual branches.
* `qkv_bias`: whether the query, key, and value projections have a bias.
  The original GPT-2 has these biases; here they are disabled, as in the
  book, for a simpler model.

In [2]:
# Dimensions of GPT-2 Small
GPT_CONFIG_SMALL = {
    "vocab_size": 50257,  # Vocabulary size of the GPT-2 BPE tokenizer
    "context_length": 1024,  # Maximum number of input tokens
    "emb_dim": 768,  # Embedding dimension
    "n_heads": 12,  # Number of attention heads
    "n_layers": 12,  # Number of transformer blocks
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": False,  # Bias in the query, key, and value projections
}

### Multi-head attention

This is the efficient `MultiHeadAttention` class of
[Chapter 2](../chap2/multi_head_attention.ipynb). Attention is the only
component where tokens **exchange information**: each token builds its new
vector as a weighted average of the value vectors of itself and of the tokens
before it.

The steps:

1. project the input into queries, keys, and values, each `(b, num_tokens, d_out)`;
2. split the last dimension into `num_heads` heads of `head_dim` values each,
   and move the head dimension forward: `(b, num_heads, num_tokens, head_dim)`;
3. compute the scores of every query with every key, one matrix per head:
   `(b, num_heads, num_tokens, num_tokens)`;
4. set the scores of future tokens to `-inf` with the causal mask, so that
   softmax gives them a weight of exactly 0;
5. scale the scores by `sqrt(head_dim)` and normalize each row with softmax;
6. apply dropout to the attention weights, during training only;
7. multiply the weights by the values, and concatenate the heads back into
   `(b, num_tokens, d_out)`;
8. mix the outputs of the heads with the output projection `out_proj`.

The causal mask is stored with `register_buffer`: it moves with the model to
the selected device (CPU, CUDA, or MPS), but it is not a trainable parameter.

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        # The size of each head: d_out is split evenly across the heads.
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # Combines the outputs of the heads; GPT-2 has this layer too.
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        # 1 above the diagonal marks the future tokens that must be hidden.
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Dot products of every query with every key, for each head at once:
        # (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Hide the future tokens: exp(-inf) = 0, so softmax gives them weight 0
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Scale by sqrt(head_dim), then normalize each row over the keys
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # Shape: (b, num_tokens, d_out)

        return context_vec

### Layer normalization

Layer normalization rescales **each token vector on its own**: over its
`emb_dim` values, it subtracts the mean and divides by the standard deviation,
so that the result has mean 0 and variance 1. Then two trainable vectors,
`scale` and `shift`, let the model restore the scale and offset that work best.

```text
LayerNorm(x) = scale * (x - mean(x)) / sqrt(var(x) + eps) + shift
```

Why it helps: through many stacked layers, the values of the activations can
grow or shrink a lot, which makes training slow and unstable. Normalizing
keeps them in a predictable range.

Details of the implementation:

* `dim=-1` normalizes over the embedding dimension, so every token of every
  sequence is normalized independently. Unlike batch normalization, the result
  does not depend on the other examples of the batch.
* `unbiased=False` divides the variance by `n` instead of `n - 1`, as GPT-2 and
  `torch.nn.LayerNorm` do. With 768 values, the difference is tiny, but it
  matters to reproduce the GPT-2 weights exactly.
* `eps` avoids a division by zero when a vector has zero variance.

In [4]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        # Statistics of each token vector, over its emb_dim values
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

### The activation function: GELU

Without a non-linear activation, two linear layers in a row are equivalent to a
single linear layer. GPT-2 uses **GELU** (Gaussian Error Linear Unit):

```text
GELU(x) = x * Phi(x)
```

where `Phi` is the cumulative distribution function of the standard normal
distribution. Compared with ReLU, `max(0, x)`:

* GELU is **smooth**, with no corner at 0, which makes optimization easier;
* for negative inputs it is not exactly zero: small negative values pass with a
  small negative output, so their gradient is not zero.

For large positive `x`, GELU is close to `x`; for large negative `x`, it is
close to 0. The class below uses the **tanh approximation** of the original
GPT-2, the same as `torch.nn.GELU(approximate="tanh")`.

In [5]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # tanh approximation of x * Phi(x), as in the original GPT-2
        coefficient = torch.sqrt(torch.tensor(2.0 / torch.pi))
        inner = coefficient * (x + 0.044715 * torch.pow(x, 3))
        return 0.5 * x * (1 + torch.tanh(inner))

### The feed-forward network

The feed-forward network processes **each token separately**, with the same
weights for all positions: it does not mix information between tokens, which
is the job of attention. It has two linear layers with GELU in between:

* the first expands each vector from `emb_dim` to `4 * emb_dim` values
  (768 to 3072);
* the second contracts it back to `emb_dim`.

The wider hidden layer gives the model room to compute richer features of each
token. The feed-forward network holds most of the parameters of a transformer
block: about 4.7M, against about 2.4M for attention, with `emb_dim = 768`.

In [6]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),  # Expand: D -> 4D
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),  # Contract: 4D -> D
        )

    def forward(self, x):
        return self.layers(x)

### The transformer block

The transformer block is the unit that GPT stacks `n_layers` times. It has two
sub-layers, each wrapped in the same pattern:

```text
x = x + Dropout(MultiHeadAttention(LayerNorm(x)))   # tokens exchange information
x = x + Dropout(FeedForward(LayerNorm(x)))          # each token is processed on its own
```

* **Residual (shortcut) connections.** Each sub-layer computes a *change* that
  is added to its input, instead of replacing it. The gradient can flow back
  through the additions without passing through every layer, so deep stacks
  remain trainable. Each block refines the token vectors, rather than
  rewriting them.
* **Pre-LayerNorm.** The normalization is applied *before* each sub-layer,
  inside the residual branch, as in GPT-2. The original transformer normalized
  after the addition; normalizing before makes the training of deep models
  more stable.
* **Dropout on the residual branch.** During training, dropout is applied to
  the output of each sub-layer, before the addition.

The output has the same shape as the input, `(batch_size, num_tokens, emb_dim)`.

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut (residual) connection around the attention sub-layer
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape: (batch_size, num_tokens, emb_dim)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut (residual) connection around the feed-forward sub-layer
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

### The GPT model

The complete model has the same structure as `DummyGPTModel`, now with real
components:

1. the token embedding and the position embedding, added together;
2. dropout on the embeddings;
3. the stack of `n_layers` transformer blocks;
4. a final layer normalization, needed because with pre-LayerNorm the output
   of the last block is not normalized;
5. the output head, a linear layer from `emb_dim` to `vocab_size`.

The result is a tensor of **logits**, `(batch_size, num_tokens, vocab_size)`:
for every position, one score per vocabulary token, which softmax turns into
the distribution of the next token. See
[What are the logits produced by the GPT model?](../docs/questions-and-answers.md#what-are-the-logits-produced-by-the-gpt-model)

The number of input tokens cannot exceed `context_length`: the position
embedding table has no rows for longer sequences.

In [8]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)  # Shape: (batch_size, seq_len, emb_dim)
        # One position vector per position, broadcast over the batch: (seq_len, emb_dim)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape: (batch_size, seq_len, emb_dim)

        x = self.drop_emb(x)

        x = self.trf_blocks(x)

        x = self.final_norm(x)

        logits = self.out_head(x)  # Shape: (batch_size, seq_len, vocab_size)

        return logits

### Count the parameters

Now the model is complete, so `count_parameters` counts the parameters that
really exist in the instance, and `estimate_gpt_parameters` computes them
from the configuration alone. The two numbers must agree.

The model is created on PyTorch's `meta` device, which records the shapes of the
tensors without allocating their values: counting a model of this size needs
almost no memory. A model on the `meta` device cannot compute outputs.

In [9]:
from build_llm.model_parameters import count_parameters, estimate_gpt_parameters

# This instance is only for counting, not for running a numerical forward pass.
with torch.device("meta"):
    model = GPTModel(GPT_CONFIG_SMALL)

print(f"Parameters currently present in GPTModel: {count_parameters(model):,}")
print(f"Trainable parameters: {count_parameters(model, trainable_only=True):,}")

full_model_counts = estimate_gpt_parameters(GPT_CONFIG_SMALL)

print()
print("Estimated full GPT parameters (separate input and output weights):")
for component, parameter_count in full_model_counts.items():
    print(f"{component:>20}: {parameter_count:>15,}")

Parameters currently present in GPTModel: 163,009,536
Trainable parameters: 163,009,536

Estimated full GPT parameters (separate input and output weights):
    token_embeddings:      38,597,376
 position_embeddings:         786,432
           attention:      28,320,768
        feed_forward:      56,669,184
         block_norms:          36,864
          final_norm:           1,536
         output_head:      38,597,376
               total:     163,009,536


### Why about 163M parameters, and not the 124M of GPT-2 Small?

The model has the dimensions of GPT-2 Small, but about 163M parameters,
while GPT-2 Small is known as a 124M model. Two choices explain the difference:

* **No weight tying.** GPT-2 reuses the token embedding matrix as the weight of
  the output head, because both have `vocab_size × emb_dim` values. Here the
  output head has its own matrix, which adds 38,597,376 parameters.
* **No query, key, and value biases.** `qkv_bias=False` removes
  `3 × 768` parameters per block, 27,648 in total.

With tied weights and `qkv_bias=True`, the same formula gives 124,439,808
parameters (see [`dummy_gpt.ipynb`](dummy_gpt.ipynb)).

The breakdown also shows where the parameters are. The token embeddings and
the output head alone hold almost half of the model, because the vocabulary is
large. In the transformer blocks, the feed-forward networks have twice the
parameters of attention.

### Summary

* A GPT model is a stack of identical transformer blocks between the
  embeddings and the output head.
* Attention mixes information between tokens; the feed-forward network
  processes each token on its own.
* Layer normalization and residual connections keep the training of deep stacks
  stable.
* Every block preserves the shape `(batch_size, num_tokens, emb_dim)`; the
  output head maps it to logits of shape `(batch_size, num_tokens, vocab_size)`.
* The model is still untrained: its weights are random, so the logits are not
  meaningful yet.